# Benchmark insertion of new predictions
1. Insert 1B rows monotonic ids: ~0.49 hrs
2. Insert 1B rows non-monotonic ids
3. Insert 1000 random rows w/ fk: ~41ms, ~20k r/s
4. Insert 1000 sequential rows w/ fk: 
5. Insert 1000 rows w/ ij index: 
6. Insert 1000 rows w/ z index

## Test run: 1 million rows in the primary table, 100,000 rows in the foreign key table.

In [ ]:
import psycopg2
import time

DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=postgres-test"
TABLE_1_NAME = "proto_2_1_uuid_primary"
TABLE_2_NAME = "proto_2_1_uuid_foreign"
TOTAL_ROWS = 1_000_000
TOTAL_FK_ROWS = 100_000
BATCH_SIZE = 100_000
COMMIT_FREQUENCY = 100

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

print("Creating tables...")
cur.execute(f"DROP TABLE IF EXISTS {TABLE_2_NAME} CASCADE;")
cur.execute(f"DROP TABLE IF EXISTS {TABLE_1_NAME} CASCADE;")

# Create table with BIGSERIAL primary key
cur.execute(f"""
CREATE TABLE {TABLE_1_NAME} (
    id BIGSERIAL PRIMARY KEY,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE {TABLE_2_NAME} (
    id BIGSERIAL PRIMARY KEY,
    id_ref BIGINT NOT NULL REFERENCES {TABLE_1_NAME}(id),
    data INT NOT NULL,
    value FLOAT NOT NULL
);
""")
conn.commit()

# Tune PostgreSQL for bulk inserts
cur.execute("SET maintenance_work_mem = '2GB';")
cur.execute("SET work_mem = '512MB';")
cur.execute("SET synchronous_commit = OFF;")
conn.commit()

print("\n=== Phase 1: Insert Rows ===\n")

rows_inserted = 0
start_time = time.time()
batch_count = 0

try:
    cur.execute("BEGIN;")
    
    while rows_inserted < TOTAL_ROWS:
        # Insert simple rows
        cur.execute(f"""
            INSERT INTO {TABLE_1_NAME} (created_at)
            SELECT CURRENT_TIMESTAMP
            FROM generate_series(1, {BATCH_SIZE});
        """)
        
        rows_inserted += BATCH_SIZE
        batch_count += 1
        
        if batch_count % COMMIT_FREQUENCY == 0:
            conn.commit()
            elapsed = time.time() - start_time
            rate = rows_inserted / elapsed
            remaining = TOTAL_ROWS - rows_inserted
            eta_sec = remaining / rate if rate > 0 else 0
            print(f"Inserted {rows_inserted:,} rows in {elapsed:.2f}s ({rate:.0f} rows/sec) - ETA: {eta_sec/3600:.2f}h")
            cur.execute("BEGIN;")
    
    conn.commit()
    
except Exception as e:
    conn.rollback()
    print(f"Error Phase 1: {e}")
    raise

elapsed_phase1 = time.time() - start_time
rate_phase1 = rows_inserted / elapsed_phase1

print(f"\nPhase 1 Complete:")
print(f"  Total rows: {rows_inserted:,}")
print(f"  Elapsed time: {elapsed_phase1:.2f}s")
print(f"  Throughput: {rate_phase1:.0f} rows/sec")
print(f"  Est. time for 1B: {1_000_000_000 / rate_phase1 / 3600:.2f} hours")

print("\n=== Phase 2: Insert FK Rows (Sequential) ===\n")

rows_inserted_fk = 0
start_time_fk = time.time()
batch_count_fk = 0

try:
    cur.execute("BEGIN;")
    
    while rows_inserted_fk < TOTAL_FK_ROWS:
        # Sample from primary table sequentially
        cur.execute(f"""
            INSERT INTO {TABLE_2_NAME} (id_ref, data, value)
            SELECT 
                id,
                row_number() OVER() % 1000,
                (row_number() OVER() - 1) * 0.5
            FROM (
                SELECT id FROM {TABLE_1_NAME} 
                ORDER BY id
                LIMIT {BATCH_SIZE}
                OFFSET {rows_inserted_fk}
            ) t;
        """)
        
        rows_inserted_fk += BATCH_SIZE
        batch_count_fk += 1
        
        if batch_count_fk % COMMIT_FREQUENCY == 0:
            conn.commit()
            elapsed = time.time() - start_time_fk
            rate = rows_inserted_fk / elapsed
            print(f"Inserted {rows_inserted_fk:,} rows in {elapsed:.2f}s ({rate:.0f} rows/sec)")
            cur.execute("BEGIN;")
    
    conn.commit()
        
    print("Creating index...")
    start_index_time = time.time()
    cur.execute(f"CREATE INDEX idx_id_ref ON {TABLE_2_NAME}(id_ref);")
    conn.commit()
    elapsed_index = time.time() - start_index_time
    print(f"Index created in {elapsed_index:.2f}s")
    conn.commit()
    
except Exception as e:
    conn.rollback()
    print(f"Error Phase 2: {e}")
    raise

elapsed_phase2 = time.time() - start_time_fk
rate_phase2 = rows_inserted_fk / elapsed_phase2 if elapsed_phase2 > 0 else 0

print(f"\nPhase 2 Complete:")
print(f"  Total rows: {rows_inserted_fk:,}")
print(f"  Elapsed time: {elapsed_phase2:.2f}s")
print(f"  Throughput: {rate_phase2:.0f} rows/sec")

total_elapsed = elapsed_phase1 + elapsed_phase2
total_rows = rows_inserted + rows_inserted_fk
overall_rate = total_rows / total_elapsed

print("\n" + "="*60)
print("BENCHMARK SUMMARY")
print("="*60)
print(f"Phase 1 (Insert): {rows_inserted:,} rows in {elapsed_phase1:.2f}s ({rate_phase1:.0f} rows/sec)")
print(f"Phase 2 (Insert): {rows_inserted_fk:,} rows in {elapsed_phase2:.2f}s ({rate_phase2:.0f} rows/sec)")
print(f"\nTotal: {total_rows:,} rows in {total_elapsed:.2f}s ({overall_rate:.0f} rows/sec)")
print(f"Est. time for 1B rows (Phase 1): {1_000_000_000 / rate_phase1 / 3600:.2f} hours")
print("="*60)

cur.close()
conn.close()


## Insert 1B rows monotonic ids

In [ ]:
import psycopg2
import time
import random

DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=postgres-test"
TABLE_1_NAME = "b_rows_monotonic_ids"
TABLE_2_NAME = "onek_rows_with_fk"
TOTAL_ROWS = 1_000_000_000       # 1 billion primary rows
BATCH_SIZE = 100_000
COMMIT_FREQUENCY = 100
PREDICTION_BATCH_SIZE = 1_000    # batch size for the FK insert benchmark
PREDICTION_RUNS = 10             # repeat N times to get a stable average

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# Create tables only if they don't already exist
cur.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_1_NAME} (
    id BIGSERIAL PRIMARY KEY,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS {TABLE_2_NAME} (
    id BIGSERIAL PRIMARY KEY,
    id_ref BIGINT NOT NULL REFERENCES {TABLE_1_NAME}(id),
    data INT NOT NULL,
    value FLOAT NOT NULL
);
""")
conn.commit()

# Tune PostgreSQL for bulk inserts
cur.execute("SET maintenance_work_mem = '4GB';")
cur.execute("SET work_mem = '1GB';")
cur.execute("SET synchronous_commit = OFF;")
conn.commit()

# Check whether the primary table already has data
cur.execute(f"SELECT COUNT(*) FROM {TABLE_1_NAME};")
table_1_row_count = cur.fetchone()[0]

# ─────────────────────────────────────────────
# Phase 1: Insert 1 billion rows (primary table)
# ─────────────────────────────────────────────
if table_1_row_count > 0:
    print(f"Skipping Phase 1 — {TABLE_1_NAME} already exists with {table_1_row_count:,} rows.")
    rows_inserted = table_1_row_count
    elapsed_phase1 = 0
    rate_phase1 = float('nan')
else:
    print("\n=== Phase 1: Insert 1 Billion Rows (Primary Table) ===\n")

    rows_inserted = 0
    start_time = time.time()
    batch_count = 0

    try:
        cur.execute("BEGIN;")

        while rows_inserted < TOTAL_ROWS:
            cur.execute(f"""
                INSERT INTO {TABLE_1_NAME} (created_at)
                SELECT CURRENT_TIMESTAMP
                FROM generate_series(1, {BATCH_SIZE});
            """)

            rows_inserted += BATCH_SIZE
            batch_count += 1

            if batch_count % COMMIT_FREQUENCY == 0:
                conn.commit()
                elapsed = time.time() - start_time
                rate = rows_inserted / elapsed
                remaining = TOTAL_ROWS - rows_inserted
                eta_sec = remaining / rate if rate > 0 else 0
                print(
                    f"  {rows_inserted:>15,} rows | {elapsed:>8.2f}s | "
                    f"{rate:>10,.0f} rows/s | ETA: {eta_sec / 3600:.2f}h"
                )
                cur.execute("BEGIN;")

        conn.commit()

    except Exception as e:
        conn.rollback()
        print(f"Error Phase 1: {e}")
        raise

    elapsed_phase1 = time.time() - start_time
    rate_phase1 = rows_inserted / elapsed_phase1

    print(f"\nPhase 1 Complete:")
    print(f"  Total rows : {rows_inserted:,}")
    print(f"  Elapsed    : {elapsed_phase1:.2f}s")
    print(f"  Throughput : {rate_phase1:,.0f} rows/sec")


Skipping Phase 1 — proto_2_2_bigserial_primary_full already exists with 1,000,000,000 rows.

=== Phase 2: Insert 1,000 FK Rows into Empty Table (×10 runs) ===

  Run  1/10: 40.54 ms  (24,669 rows/sec)
  Run  2/10: 32.58 ms  (30,692 rows/sec)
  Run  3/10: 32.92 ms  (30,377 rows/sec)
  Run  4/10: 35.08 ms  (28,502 rows/sec)
  Run  5/10: 34.35 ms  (29,108 rows/sec)
  Run  6/10: 33.49 ms  (29,859 rows/sec)
  Run  7/10: 32.60 ms  (30,678 rows/sec)
  Run  8/10: 31.63 ms  (31,611 rows/sec)
  Run  9/10: 44.85 ms  (22,297 rows/sec)
  Run 10/10: 43.81 ms  (22,825 rows/sec)

  Avg latency : 36.19 ms
  Min latency : 31.63 ms
  Max latency : 44.85 ms
  Avg rate    : 27,635 rows/sec

BENCHMARK SUMMARY
Phase 1 (Primary, 1B rows)       : skipped (1,000,000,000 rows pre-existing)
Phase 2 (1k FK rows, empty table) : avg 36.19 ms  |  27,635 rows/sec
  Min: 31.63 ms  |  Max: 44.85 ms


## Insert 1B rows non-monotonic ids

In [ ]:
import psycopg2
import time
import random

DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=postgres-test"
TABLE_1_NAME = "b_rows_non_monotonic_ids"
TOTAL_ROWS = 1_000_000_000       # 1 billion primary rows
BATCH_SIZE = 100_000
COMMIT_FREQUENCY = 100
PREDICTION_BATCH_SIZE = 1_000    # batch size for the FK insert benchmark
PREDICTION_RUNS = 10             # repeat N times to get a stable average

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# Create tables only if they don't already exist
cur.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_1_NAME} (
    id BIGINT PRIMARY KEY,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

""")
conn.commit()

# Tune PostgreSQL for bulk inserts
cur.execute("SET maintenance_work_mem = '4GB';")
cur.execute("SET work_mem = '1GB';")
cur.execute("SET synchronous_commit = OFF;")
conn.commit()

# Check whether the primary table already has data
cur.execute(f"SELECT COUNT(*) FROM {TABLE_1_NAME};")
table_1_row_count = cur.fetchone()[0]

# ─────────────────────────────────────────────
# Phase 1: Insert 1 billion rows (primary table)
# ─────────────────────────────────────────────
if table_1_row_count > 0:
    print(f"Skipping Phase 1 — {TABLE_1_NAME} already exists with {table_1_row_count:,} rows.")
    rows_inserted = table_1_row_count
    elapsed_phase1 = 0
    rate_phase1 = float('nan')
else:
    print("\n=== Phase 1: Insert 1 Billion Rows (Primary Table) ===\n")

    rows_inserted = 0
    start_time = time.time()
    batch_count = 0

    try:
        cur.execute("BEGIN;")

        while rows_inserted < TOTAL_ROWS:
            cur.execute(f"""
                INSERT INTO {TABLE_1_NAME} (created_at)
                SELECT CURRENT_TIMESTAMP
                FROM generate_series(1, {BATCH_SIZE});
            """)

            rows_inserted += BATCH_SIZE
            batch_count += 1

            if batch_count % COMMIT_FREQUENCY == 0:
                conn.commit()
                elapsed = time.time() - start_time
                rate = rows_inserted / elapsed
                remaining = TOTAL_ROWS - rows_inserted
                eta_sec = remaining / rate if rate > 0 else 0
                print(
                    f"  {rows_inserted:>15,} rows | {elapsed:>8.2f}s | "
                    f"{rate:>10,.0f} rows/s | ETA: {eta_sec / 3600:.2f}h"
                )
                cur.execute("BEGIN;")

        conn.commit()

    except Exception as e:
        conn.rollback()
        print(f"Error Phase 1: {e}")
        raise

    elapsed_phase1 = time.time() - start_time
    rate_phase1 = rows_inserted / elapsed_phase1

    print(f"\nPhase 1 Complete:")
    print(f"  Total rows : {rows_inserted:,}")
    print(f"  Elapsed    : {elapsed_phase1:.2f}s")
    print(f"  Throughput : {rate_phase1:,.0f} rows/sec")


## Insert 1000 random rows w/ fk

In [ ]:

# ──────────────────────────────────────────────────────────────────────────────
# Phase 2: Benchmark — insert 1,000 FK rows into empty TABLE_2 (repeated N times)
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n=== Phase 2: Insert {PREDICTION_BATCH_SIZE:,} FK Rows into Empty Table (×{PREDICTION_RUNS} runs) ===\n")

latencies = []

for run in range(1, PREDICTION_RUNS + 1):
    # Truncate the FK table to ensure each run starts from empty
    cur.execute(f"TRUNCATE TABLE {TABLE_2_NAME} RESTART IDENTITY;")
    conn.commit()

    t0 = time.time()
    cur.execute(f"""
        INSERT INTO {TABLE_2_NAME} (id_ref, data, value)
        SELECT
            floor(random() * {rows_inserted})::BIGINT + 1,
            (random() * 1000)::INT,
            random() * 100.0
        FROM generate_series(1, {PREDICTION_BATCH_SIZE});
    """)
    conn.commit()
    latency_ms = (time.time() - t0) * 1000
    latencies.append(latency_ms)
    print(f"  Run {run:>2}/{PREDICTION_RUNS}: {latency_ms:.2f} ms  ({PREDICTION_BATCH_SIZE / (latency_ms / 1000):,.0f} rows/sec)")

avg_ms   = sum(latencies) / len(latencies)
min_ms   = min(latencies)
max_ms   = max(latencies)
avg_rate = PREDICTION_BATCH_SIZE / (avg_ms / 1000)

print(f"\n  Avg latency : {avg_ms:.2f} ms")
print(f"  Min latency : {min_ms:.2f} ms")
print(f"  Max latency : {max_ms:.2f} ms")
print(f"  Avg rate    : {avg_rate:,.0f} rows/sec")


## Insert 1000 sequential rows w/ fk

In [ ]:
import psycopg2
import time
import random

DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=postgres-test"
TABLE_1_NAME = "b_rows_monotonic_ids"
TABLE_2_NAME_SEQ = "onek_rows_with_fk_sequential"
PREDICTION_BATCH_SIZE = 1_000
PREDICTION_RUNS = 10

conn_seq = psycopg2.connect(DATABASE_URL)
cur_seq = conn_seq.cursor()

# Get the current row count of the primary table
cur_seq.execute(f"SELECT COUNT(*) FROM {TABLE_1_NAME};")
rows_in_primary = cur_seq.fetchone()[0]

# Create FK table if it doesn't exist
cur_seq.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_2_NAME_SEQ} (
    id BIGSERIAL PRIMARY KEY,
    id_ref BIGINT NOT NULL REFERENCES {TABLE_1_NAME}(id),
    data INT NOT NULL,
    value FLOAT NOT NULL
);
""")
conn_seq.commit()

print(f"\n=== Benchmark: Insert {PREDICTION_BATCH_SIZE:,} Sequential FK Rows (×{PREDICTION_RUNS} runs) ===\n")

seq_latencies = []

for run in range(1, PREDICTION_RUNS + 1):
    cur_seq.execute(f"TRUNCATE TABLE {TABLE_2_NAME_SEQ} RESTART IDENTITY;")
    conn_seq.commit()

    # Pick a random start offset for variety across runs
    start_id = random.randint(1, max(1, rows_in_primary - PREDICTION_BATCH_SIZE))

    t0 = time.time()
    cur_seq.execute(f"""
        INSERT INTO {TABLE_2_NAME_SEQ} (id_ref, data, value)
        SELECT
            {start_id}::BIGINT + gs - 1,
            (random() * 1000)::INT,
            random() * 100.0
        FROM generate_series(1, {PREDICTION_BATCH_SIZE}) gs;
    """)
    conn_seq.commit()
    latency_ms = (time.time() - t0) * 1000
    seq_latencies.append(latency_ms)
    print(f"  Run {run:>2}/{PREDICTION_RUNS}: {latency_ms:.2f} ms  ({PREDICTION_BATCH_SIZE / (latency_ms / 1000):,.0f} rows/sec)  [start_id={start_id:,}]")

seq_avg_ms   = sum(seq_latencies) / len(seq_latencies)
seq_min_ms   = min(seq_latencies)
seq_max_ms   = max(seq_latencies)
seq_avg_rate = PREDICTION_BATCH_SIZE / (seq_avg_ms / 1000)

print(f"\n  Avg latency : {seq_avg_ms:.2f} ms")
print(f"  Min latency : {seq_min_ms:.2f} ms")
print(f"  Max latency : {seq_max_ms:.2f} ms")
print(f"  Avg rate    : {seq_avg_rate:,.0f} rows/sec")

cur_seq.close()
conn_seq.close()


In [ ]:
# ─────────────────
# Summary
# ─────────────────
print("\n" + "=" * 70)
print("BENCHMARK SUMMARY")
print("=" * 70)
phase1_summary = (
    f"skipped ({rows_inserted:,} rows pre-existing)"
    if elapsed_phase1 == 0
    else f"{elapsed_phase1:.2f}s  |  {rate_phase1:,.0f} rows/sec"
)
print(f"Phase 1 (Primary, 1B rows)                : {phase1_summary}")
print(f"Phase 2 (1k random FK rows, empty table)  : avg {avg_ms:.2f} ms  |  {avg_rate:,.0f} rows/sec")
print(f"  Min: {min_ms:.2f} ms  |  Max: {max_ms:.2f} ms")
print(f"Phase 3 (1k sequential FK rows, empty tbl): avg {seq_avg_ms:.2f} ms  |  {seq_avg_rate:,.0f} rows/sec")
print(f"  Min: {seq_min_ms:.2f} ms  |  Max: {seq_max_ms:.2f} ms")
print("=" * 70)

cur.close()
conn.close()


## Insert 1000 rows w/ random ij index

## Insert 1000 rows w/ random z index